# Black Summer — Computing PR from CMIP6 (hist vs hist-nat)

Compute the Probability Ratio (PR) for extreme fire-season heat in SE Australia by comparing CMIP6 `historical` (factual, all forcings) vs `hist-nat` (counterfactual, natural forcing only) runs.

**Variable**: `tasmax` — monthly mean of daily maximum temperature  
**Metric**: Oct–Mar seasonal maximum `tasmax` anomaly over SE Australia  
**Models**: BCC-CSM2-MR, GFDL-ESM4, IPSL-CM6A-LR (×10 members), MRI-ESM2-0  
**Validation target**: WWA reported PR ≥ 10 for the heat component of Black Summer  

## Method

1. For each model and experiment, stream SE Australia `tasmax` from pangeo
2. Aggregate to Oct–Mar seasonal maximum
3. Compute anomalies relative to each model's own 1961–1990 climatology (removes inter-model bias)
4. Pool anomalies across all model-members per experiment
5. Fit Gaussian distributions to both pools
6. Compute PR at multiple thresholds; bootstrap uncertainty
7. Update Black Summer liability with CMIP6-derived PR

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import intake
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from scipy.stats import norm
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

PROC = Path('../../data/processed')
FIGS = Path('../../outputs/figures')

# SE Australia bounding box
LAT_S, LAT_N = -44, -28
LON_W, LON_E = 138, 154

# Climatology baseline for anomaly normalisation
CLIM = slice('1961', '1990')

# Models with both historical and hist-nat tasmax on pangeo
MODELS = ['BCC-CSM2-MR', 'GFDL-ESM4', 'IPSL-CM6A-LR', 'MRI-ESM2-0']

# Per-store timeout in seconds — skip any store that hangs longer than this
STORE_TIMEOUT = 300

## 1. Catalog query

In [ ]:
CATALOG_LOCAL = Path('../../data/processed/pangeo-cmip6.json').resolve()

print(f'Loading catalog from local cache: {CATALOG_LOCAL}')
col = intake.open_esm_datastore(str(CATALOG_LOCAL))
print('Catalog loaded.')

cat_hist = col.search(
    variable_id='tasmax', experiment_id='historical',
    table_id='Amon', source_id=MODELS,
)
cat_nat = col.search(
    variable_id='tasmax', experiment_id='hist-nat',
    table_id='Amon', source_id=MODELS,
)

print(f'historical entries: {len(cat_hist.df)}')
print(f'hist-nat entries:   {len(cat_nat.df)}')
print()
print('hist-nat members per model:')
print(cat_nat.df.groupby('source_id')['member_id'].apply(list).to_string())

## 2. Helper functions

In [ ]:
def au_area_mean(zstore, lat_s=LAT_S, lat_n=LAT_N, lon_w=LON_W, lon_e=LON_E):
    """Stream SE Australia area-weighted tasmax from a zarr store."""
    ds = xr.open_zarr(zstore, consolidated=True)
    da = ds['tasmax']

    # Drop extra dimensions (member_id, dcpp_init_year, etc.)
    extra = [d for d in da.dims if d not in ('time', 'lat', 'lon')]
    if extra:
        da = da.isel({d: 0 for d in extra})

    # Normalise longitudes to 0–360
    if da.lon.values.min() < 0:
        da = da.assign_coords(lon=(da.lon % 360)).sortby('lon')

    da = da.sel(lat=slice(lat_s, lat_n), lon=slice(lon_w % 360, lon_e % 360))
    weights = np.cos(np.deg2rad(da.lat)).broadcast_like(da)
    return da.weighted(weights).mean(dim=['lat', 'lon']).squeeze().load()


def fire_season_max(ts):
    """Oct–Mar seasonal maximum tasmax. Returns annual series indexed by Oct year."""
    # Shift so Oct–Mar aligns to a single 'year' label
    ts_shifted = ts.convert_calendar('standard', use_cftime=False) if hasattr(ts, 'convert_calendar') else ts
    df = ts_shifted.to_series()
    df.index = pd.to_datetime(df.index)
    # Label each Oct-Mar window by the October year
    df_shifted = df.copy()
    df_shifted.index = df_shifted.index - pd.DateOffset(months=9)
    return df_shifted.resample('YE').max().dropna()


def anomaly(series, clim_slice=CLIM):
    """Subtract 1961–1990 climatological mean."""
    clim_mean = series[clim_slice].mean()
    return series - clim_mean

## 3. Stream data and build anomaly pools

One zarr store at a time. Each model-member produces one time series of fire-season temperature anomalies.

In [ ]:
hist_anomalies = []  # list of annual anomaly series
nat_anomalies  = []

def _process_store(zstore):
    """Load one zarr store and return fire-season anomaly array. Runs in thread for timeout."""
    ts = au_area_mean(zstore)
    seasonal = fire_season_max(ts)
    return anomaly(seasonal)

for label, cat, store in [('historical', cat_hist, hist_anomalies),
                           ('hist-nat',   cat_nat,  nat_anomalies)]:
    print(f'\n--- {label} ({len(cat.df)} entries) ---')
    for _, row in cat.df.iterrows():
        tag = f"{row['source_id']} {row['member_id']}"
        print(f'  {tag}...', end=' ', flush=True)
        try:
            with ThreadPoolExecutor(max_workers=1) as ex:
                future = ex.submit(_process_store, row['zstore'])
                anom = future.result(timeout=STORE_TIMEOUT)
            store.append(anom.values)
            print(f'n={len(anom)} years, mean anom={anom.mean():.2f} K')
        except FuturesTimeoutError:
            print(f'TIMEOUT (>{STORE_TIMEOUT}s) — skipping')
        except Exception as e:
            print(f'FAILED: {e}')

hist_pool = np.concatenate(hist_anomalies)
nat_pool  = np.concatenate(nat_anomalies)
print(f'\nPooled samples — historical: {len(hist_pool)}, hist-nat: {len(nat_pool)}')

## 4. Distribution fitting and PR calculation

Fit Gaussian distributions to each pool. Compute PR at multiple thresholds to show robustness.

In [ ]:
# Fit Gaussian to each pool
mu_hist, sigma_hist = norm.fit(hist_pool)
mu_nat,  sigma_nat  = norm.fit(nat_pool)

print('Fitted distributions (fire-season tasmax anomaly, °C):')
print(f'  historical: μ={mu_hist:.3f}  σ={sigma_hist:.3f}')
print(f'  hist-nat:   μ={mu_nat:.3f}  σ={sigma_nat:.3f}')
print(f'  Shift in mean: {mu_hist - mu_nat:.3f} °C  (warming attributable to anthropogenic forcing)')

# PR at a range of thresholds
# The observed 2019/2020 fire season was ~1.5–2°C above the 1961-1990 mean
# Use the 90th–99th percentile of the historical distribution as thresholds
percentiles = [90, 95, 97, 99]
thresholds  = [np.percentile(hist_pool, p) for p in percentiles]

print()
print(f'{'Threshold':>12}  {'pct':>5}  {'P1 (hist)':>10}  {'P0 (nat)':>10}  {'PR':>8}  {'FAR':>8}')
pr_results = []
for pct, thresh in zip(percentiles, thresholds):
    p1  = 1 - norm.cdf(thresh, mu_hist, sigma_hist)
    p0  = 1 - norm.cdf(thresh, mu_nat,  sigma_nat)
    pr  = p1 / p0 if p0 > 0 else np.inf
    far = 1 - 1/pr if pr > 1 else 0
    pr_results.append({'pct': pct, 'threshold_degC': thresh, 'p1': p1, 'p0': p0, 'pr': pr, 'far': far})
    print(f'  {thresh:>10.2f}°C  {pct:>5}  {p1:>10.4f}  {p0:>10.4f}  {pr:>8.1f}  {far:>8.3f}')

pr_df = pd.DataFrame(pr_results)

In [ ]:
# Bootstrap uncertainty on PR at the 97th percentile (closest to observed 2019/2020)
THRESH_PCT = 97
thresh_97  = np.percentile(hist_pool, THRESH_PCT)
N_BOOT     = 2000

np.random.seed(42)
boot_pr = []
for _ in range(N_BOOT):
    h = np.random.choice(hist_pool, size=len(hist_pool), replace=True)
    n = np.random.choice(nat_pool,  size=len(nat_pool),  replace=True)
    mh, sh = norm.fit(h)
    mn, sn = norm.fit(n)
    p1 = 1 - norm.cdf(thresh_97, mh, sh)
    p0 = 1 - norm.cdf(thresh_97, mn, sn)
    if p0 > 0:
        boot_pr.append(p1 / p0)

pr_median = np.median(boot_pr)
pr_p05    = np.percentile(boot_pr, 5)
pr_p95    = np.percentile(boot_pr, 95)

print(f'PR at {THRESH_PCT}th percentile threshold ({thresh_97:.2f}°C anomaly):')
print(f'  Median: {pr_median:.1f}')
print(f'  5–95th: [{pr_p05:.1f}, {pr_p95:.1f}]')
print(f'  FAR:    {1 - 1/pr_median:.3f}')
print()
print(f'WWA published PR (heat component): ≥10 (lower bound)')
print(f'Agreement: {"good" if pr_median >= 5 else "lower than expected — models likely underestimate"}')

## 5. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: distribution comparison
ax = axes[0]
x = np.linspace(min(hist_pool.min(), nat_pool.min()) - 0.5,
                max(hist_pool.max(), nat_pool.max()) + 0.5, 300)

ax.hist(nat_pool,  bins=40, density=True, alpha=0.4, color='#2196F3', label='hist-nat (counterfactual)')
ax.hist(hist_pool, bins=40, density=True, alpha=0.4, color='#FF5722', label='historical (factual)')
ax.plot(x, norm.pdf(x, mu_nat,  sigma_nat),  color='#2196F3', linewidth=2)
ax.plot(x, norm.pdf(x, mu_hist, sigma_hist), color='#FF5722', linewidth=2)
ax.axvline(thresh_97, color='k', linestyle='--', linewidth=1.5, label=f'97th pct threshold ({thresh_97:.1f}°C)')
ax.set_xlabel('Fire-season tasmax anomaly (°C, rel. 1961–1990)')
ax.set_ylabel('Density')
ax.set_title('SE Australia fire-season temperature\nhistorical vs counterfactual', fontsize=11)
ax.legend(fontsize=9)

# Right: PR vs threshold
ax2 = axes[1]
thresh_range = np.linspace(np.percentile(hist_pool, 80), np.percentile(hist_pool, 99.5), 100)
pr_curve = []
for t in thresh_range:
    p1 = 1 - norm.cdf(t, mu_hist, sigma_hist)
    p0 = 1 - norm.cdf(t, mu_nat,  sigma_nat)
    pr_curve.append(p1/p0 if p0 > 1e-10 else np.nan)

ax2.plot(thresh_range, pr_curve, color='#FF5722', linewidth=2)
ax2.axvline(thresh_97, color='k', linestyle='--', linewidth=1.5, label=f'97th pct ({thresh_97:.1f}°C)')
ax2.axhline(10, color='grey', linestyle=':', linewidth=1, label='WWA lower bound (PR=10)')
ax2.scatter([thresh_97], [pr_median], color='k', zorder=5, s=60, label=f'PR median = {pr_median:.1f}')
ax2.fill_between(
    [thresh_97 - 0.05, thresh_97 + 0.05],
    [pr_p05, pr_p05], [pr_p95, pr_p95],
    color='k', alpha=0.2, label=f'5–95th [{pr_p05:.1f}, {pr_p95:.1f}]'
)
ax2.set_xlabel('Fire-season tasmax threshold (°C anomaly)')
ax2.set_ylabel('Probability Ratio (PR)')
ax2.set_title('PR vs threshold — CMIP6 ensemble', fontsize=11)
ax2.legend(fontsize=9)
ax2.set_ylim(0, None)

plt.tight_layout()
plt.savefig(FIGS / 'black_summer_pr_cmip6.png', bbox_inches='tight')
plt.show()

## 6. Update Black Summer liability with CMIP6-derived PR

Compare three PR sources side by side: WWA published, CMIP6 central, CMIP6 conservative.

In [ ]:
lb = pd.read_parquet(PROC / 'black_summer_liability.parquet')

AUD_TO_USD = 0.69
d_central  = 10.0 * AUD_TO_USD  # direct economic, USD B

far_wwa_low     = 1 - 1/4          # WWA FWI conservative
far_wwa_central = 1 - 1/9          # WWA MSR
far_cmip6_p05   = 1 - 1/pr_p05     # CMIP6 lower bound
far_cmip6_med   = 1 - 1/pr_median  # CMIP6 median
far_cmip6_p95   = 1 - 1/pr_p95     # CMIP6 upper bound

print('FAR comparison across PR sources (direct economic damages, AUD 10B):')
print(f'  WWA FWI lower bound  (PR=4):              FAR={far_wwa_low:.3f}')
print(f'  WWA MSR central      (PR=9):              FAR={far_wwa_central:.3f}')
print(f'  CMIP6 5th pct        (PR={pr_p05:.1f}):          FAR={far_cmip6_p05:.3f}')
print(f'  CMIP6 median         (PR={pr_median:.1f}):          FAR={far_cmip6_med:.3f}')
print(f'  CMIP6 95th pct       (PR={pr_p95:.1f}):          FAR={far_cmip6_p95:.3f}')

# Add CMIP6-derived liability columns
share = lb['cm_warming_share']
lb['liability_cmip6_p05_USD_M'] = share * far_cmip6_p05 * d_central * 1000
lb['liability_cmip6_med_USD_M'] = share * far_cmip6_med * d_central * 1000
lb['liability_cmip6_p95_USD_M'] = share * far_cmip6_p95 * d_central * 1000

lb = lb.sort_values('liability_cmip6_med_USD_M', ascending=False).reset_index(drop=True)

print()
print('Top 10 entities — CMIP6-derived liability (USD M, direct economic damages):')
top10 = lb.head(10)[['parent_entity', 'liability_wwa_central_USD_M' if 'liability_wwa_central_USD_M' in lb.columns else 'liability_central_USD_M',
                      'liability_cmip6_p05_USD_M', 'liability_cmip6_med_USD_M', 'liability_cmip6_p95_USD_M']].copy()
top10.columns = ['Entity', 'WWA central $M', 'CMIP6 p05 $M', 'CMIP6 median $M', 'CMIP6 p95 $M']
for c in top10.columns[1:]:
    top10[c] = top10[c].map('{:,.1f}'.format)
print(top10.to_string(index=False))

## 7. Save outputs

In [ ]:
lb.to_parquet(PROC / 'black_summer_liability.parquet', index=False)
pr_df.to_csv(PROC / 'black_summer_pr_cmip6.csv', index=False)

# Save bootstrap distribution
pd.DataFrame({'pr_boot': boot_pr}).to_parquet(PROC / 'black_summer_pr_bootstrap.parquet', index=False)

print('Saved:')
print(f'  black_summer_liability.parquet    — updated with CMIP6-derived PR columns')
print(f'  black_summer_pr_cmip6.csv         — PR at 4 percentile thresholds')
print(f'  black_summer_pr_bootstrap.parquet — {len(boot_pr):,} bootstrap PR samples')
print()
print('CMIP6 PR summary (97th pct threshold, heat/tasmax):')
print(f'  Median: {pr_median:.1f}  [5–95th: {pr_p05:.1f}–{pr_p95:.1f}]')
print(f'  FAR:    {1-1/pr_median:.3f}')
print(f'  WWA published (heat): ≥10')

## Key findings

- **CMIP6 PR (97th pct threshold)**: 0.6 [5–95th: 0.5–0.7] — **less than 1**
- **FAR**: −0.66 (negative — these models show counterfactual world as *more* extreme than factual)
- **vs WWA**: these models give PR ≈ 0.6 vs WWA ≥ 10. The CMIP6 hist-nat verification **failed** for this model subset.
- **Distribution shift**: hist-nat pool has *higher* variance than historical, driving P0 > P1 at all thresholds
- **Root cause**: the 4 models with hist-nat tasmax available on pangeo are a non-random subset with poor Australian regional performance. IPSL-CM6A-LR contributes 10 hist-nat members vs fewer historical members, artificially inflating hist-nat variance. This is consistent with the documented CMIP6 underestimation of Australian warming (notebook 02: ensemble amplification 0.93 vs observed 1.35).
- **Implication**: CMIP6 independent verification is not viable with the currently available hist-nat tasmax subset. WWA PR values (PR=4/9/15) remain the best available estimates and are already known to be conservative lower bounds. The CMIP6 liability columns in `black_summer_liability.parquet` should **not** be used.

→ See `wiki/findings/2026-05-24-black-summer-pr-cmip6.md`